# Analyst 2: ZIP Clustering and Response-Time Analysis

This notebook covers **RQ2 + RQ3** for the Manhattan 311 project. It uses the cleaned 2025 Manhattan dataset and reuses Analyst 1's four RQ1 topic names:

- Sanitation & Property Conditions
- Residential Noise & Street Disruption
- Parking & Construction Violations
- Building Noise & Vendor Issues

Main outputs: ZIP-level KMeans clusters, cluster profile table, PCA cluster figure, response-time boxplot, and significance tests.

## Method Notes

- Unit of clustering: `incident_zip`
- KMeans features: ZIP-level shares of the four Analyst 1 RQ1 topics
- Profiling features: top complaint types, ZIP membership, topic mix, and response-time summaries
- Response time is **not** used to form clusters, so RQ3 can test whether handling time differs after clusters are assigned.
- Because `response_hours` is strongly right-skewed, Kruskal-Wallis is the main omnibus test; ANOVA on `log1p(response_hours)` is included as a robustness check.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "source" / "311_2025_manhattan_cleaned.csv").exists() else cwd.parent
DATA_PATH = PROJECT_ROOT / "source" / "311_2025_manhattan_cleaned.csv"
OUT_DIR = PROJECT_ROOT / "outputs" / "analysis2"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

In [ ]:
# Optional rerun: regenerates all Analysis 2 tables and figures.
# Leave this cell on if you want the notebook to be fully reproducible.
subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "analysis2_zip_clustering_response.py"),
    "--input", str(DATA_PATH),
    "--out-dir", str(OUT_DIR),
], check=True)

In [ ]:
# Load key outputs
cluster_profiles = pd.read_csv(TABLE_DIR / "cluster_profiles.csv")
zip_assignments = pd.read_csv(TABLE_DIR / "zip_cluster_assignments.csv", dtype={"incident_zip": str})
kmeans_diag = pd.read_csv(TABLE_DIR / "kmeans_diagnostics.csv")
response_tests = pd.read_csv(TABLE_DIR / "response_time_omnibus_tests.csv")
pairwise_tests = pd.read_csv(TABLE_DIR / "response_time_pairwise_tests.csv")
response_summary = pd.read_csv(TABLE_DIR / "response_time_summary_by_cluster.csv")

cluster_profiles

## RQ2: ZIP-Level Community Issue Profiles

The clustering result below uses KMeans with **k = 4** so that ZIP clusters can be interpreted against Analyst 1's four topic names. The diagnostic table is still shown to document the clustering tradeoff.

In [ ]:
kmeans_diag

![KMeans diagnostics](../outputs/analysis2/figures/kmeans_elbow_silhouette.png)

![ZIP clusters in PCA space](../outputs/analysis2/figures/zip_pca_clusters.png)

![Cluster and RQ1 theme alignment](../outputs/analysis2/figures/cluster_theme_heatmap.png)

![Cluster profile table](../outputs/analysis2/figures/cluster_profile_table.png)

In [ ]:
# ZIP assignments for mapping or slide tables
zip_assignments[["incident_zip", "cluster_label", "total_complaints", "pca1", "pca2"]].sort_values(["cluster_label", "incident_zip"])

## RQ3: Do Response Times Differ Across ZIP Clusters?

The response-time distribution is highly right-skewed, so the plot uses `log1p(response_hours)`. The statistical conclusion is based mainly on Kruskal-Wallis, followed by Holm-adjusted pairwise Mann-Whitney tests.

In [ ]:
response_summary

In [ ]:
response_tests

In [ ]:
pairwise_tests[["cluster_1", "cluster_2", "median_1", "median_2", "p_value", "p_value_holm", "significant_0_05"]]

![Response time boxplot](../outputs/analysis2/figures/response_time_boxplot.png)

## Short Interpretation for Report Draft

## RQ2: ZIP-Level Community Issue Profiles

K-Means was run with k=4 on ZIP-level RQ1 topic-share features. Complaint-type percentages are used for profiling, while response time is excluded from the clustering features so it can be evaluated separately in RQ3.

For k=4, inertia = 57.871 and silhouette = 0.301.

Cluster labels were assigned after profiling each cluster against Analyst 1's four RQ1 themes:

- Cluster A: Sanitation & Property Conditions: 12 ZIPs, 4850 records. Assigned RQ1 theme share: 37.0%; strongest observed topic: Sanitation & Property Conditions (37.0%). Top complaints: Illegal Parking: 9.0%; Encampment: 7.9%; Heat/Hot Water: 7.5%; Noise - Residential: 7.4%; Noise - Street/Sidewalk: 4.0%.
- Cluster B: Residential Noise & Street Disruption: 10 ZIPs, 7390 records. Assigned RQ1 theme share: 38.6%; strongest observed topic: Residential Noise & Street Disruption (38.6%). Top complaints: Heat/Hot Water: 13.7%; Noise - Residential: 13.0%; Noise - Street/Sidewalk: 12.3%; Illegal Parking: 9.2%; Noise - Commercial: 4.8%.
- Cluster C: Parking & Construction Violations: 7 ZIPs, 2452 records. Assigned RQ1 theme share: 28.9%; strongest observed topic: Sanitation & Property Conditions (32.0%). Top complaints: Illegal Parking: 11.9%; Encampment: 6.7%; Homeless Person Assistance: 5.6%; Heat/Hot Water: 5.4%; Noise: 5.3%.
- Cluster D: Building Noise & Vendor Issues: 11 ZIPs, 4051 records. Assigned RQ1 theme share: 31.8%; strongest observed topic: Building Noise & Vendor Issues (31.8%). Top complaints: Heat/Hot Water: 12.0%; Noise - Residential: 10.0%; Illegal Parking: 8.8%; Noise - Street/Sidewalk: 7.9%; Vendor Enforcement: 6.3%.

## RQ3: Response-Time Differences

Because response_hours is strongly right-skewed, the main significance test is Kruskal-Wallis. A one-way ANOVA on log1p(response_hours) is also reported as a robustness check.

Kruskal-Wallis statistic = 41.087, p-value = 6.267e-09.

Median response hours by cluster:

- Cluster A: Sanitation & Property Conditions: median=10.36, IQR=(0.85, 92.57), p90=726.74, n=4850.
- Cluster B: Residential Noise & Street Disruption: median=6.10, IQR=(0.77, 71.91), p90=416.10, n=7390.
- Cluster C: Parking & Construction Violations: median=11.53, IQR=(1.13, 87.15), p90=793.89, n=2452.
- Cluster D: Building Noise & Vendor Issues: median=5.06, IQR=(0.67, 76.68), p90=600.37, n=4051.